# 02 — Score métier custom

## Objectif

Définir une fonction de coût qui reflète les enjeux business du credit scoring :
**un Faux Négatif (FN) coûte 10× plus cher qu'un Faux Positif (FP)**.

Formule retenue : `Coût métier = 10 × FN + 1 × FP`

Plus le coût est **bas**, meilleur est le modèle pour la banque.

In [1]:
import numpy as np
from sklearn.metrics import confusion_matrix

In [2]:
# Vraies valeurs (réalité)
y_true = np.array([0, 0, 1, 1, 0])  # 1 = défaut, 0 = remboursé

# Prédictions du modèle
y_pred = np.array([0, 1, 0, 1, 0])

print("Vraies valeurs :", y_true)
print("Prédictions    :", y_pred)

Vraies valeurs : [0 0 1 1 0]
Prédictions    : [0 1 0 1 0]


In [3]:
cm = confusion_matrix(y_true, y_pred)
print("Matrice de confusion :")
print(cm)

Matrice de confusion :
[[2 1]
 [1 1]]


In [4]:
# .ravel() transforme la matrice 2x2 en 4 valeurs séparées
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print(f"TN (Vrai Négatif) : {tn}")
print(f"FP (Faux Positif) : {fp}")
print(f"FN (Faux Négatif) : {fn}")
print(f"TP (Vrai Positif) : {tp}")

TN (Vrai Négatif) : 2
FP (Faux Positif) : 1
FN (Faux Négatif) : 1
TP (Vrai Positif) : 1


## 2. Application de la formule métier

Maintenant qu'on a extrait FN et FP, on peut calculer le coût métier.

In [5]:
# Coefficients métier
COST_FN = 10  # un FN coûte 10× plus qu'un FP
COST_FP = 1

# Calcul du coût total
cout_metier = (COST_FN * fn) + (COST_FP * fp)

print(f"Coût métier total : {cout_metier}")
print(f"   = ({COST_FN} × FN) + ({COST_FP} × FP)")
print(f"   = ({COST_FN} × {fn}) + ({COST_FP} × {fp})")
print(f"   = {COST_FN * fn} + {COST_FP * fp}")
print(f"   = {cout_metier}")

Coût métier total : 11
   = (10 × FN) + (1 × FP)
   = (10 × 1) + (1 × 1)
   = 10 + 1
   = 11


In [6]:
def business_cost(y_true, y_pred, cost_fn=10, cost_fp=1):
    """
    Calcule le coût métier d'un modèle de credit scoring.
    
    Un FN (mauvais client accepté) coûte cost_fn (par défaut 10).
    Un FP (bon client refusé) coûte cost_fp (par défaut 1).
    
    Parameters
    ----------
    y_true : array-like
        Les vraies valeurs (0 ou 1)
    y_pred : array-like
        Les prédictions du modèle (0 ou 1)
    cost_fn : int, default=10
        Le coût d'un Faux Négatif
    cost_fp : int, default=1
        Le coût d'un Faux Positif
    
    Returns
    -------
    int
        Le coût métier total (plus c'est bas, mieux c'est)
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    cout = (cost_fn * fn) + (cost_fp * fp)
    return cout

In [7]:
# On retest notre mini-exemple avec la fonction
cout = business_cost(y_true, y_pred)
print(f"Coût métier : {cout}")

Coût métier : 11


## 4. Application : comparer 2 modèles

Reprenons les 2 modèles du cours :
- Modèle A : FP=50, FN=30 → coût attendu = 350
- Modèle B : FP=100, FN=5 → coût attendu = 150

On va simuler ces 2 modèles pour vérifier que notre fonction donne les bons résultats.

In [8]:
# Simulons un dataset de 1000 clients
# Réalité : 100 défauts (1) + 900 bons payeurs (0)
y_true_test = np.array([1]*100 + [0]*900)

# --- MODÈLE A ---
# Sur les 100 défauts : 70 bien détectés (TP), 30 ratés (FN)
# Sur les 900 bons : 850 bien détectés (TN), 50 refusés à tort (FP)
y_pred_A = np.array([1]*70 + [0]*30 + [1]*50 + [0]*850)

cout_A = business_cost(y_true_test, y_pred_A)
print(f"Coût métier Modèle A : {cout_A}")

Coût métier Modèle A : 350


In [9]:
# --- MODÈLE B ---
# Sur les 100 défauts : 95 bien détectés (TP), 5 ratés (FN)
# Sur les 900 bons : 800 bien détectés (TN), 100 refusés à tort (FP)
y_pred_B = np.array([1]*95 + [0]*5 + [1]*100 + [0]*800)

cout_B = business_cost(y_true_test, y_pred_B)
print(f"Coût métier Modèle B : {cout_B}")

Coût métier Modèle B : 150


In [10]:
print(f"Modèle A : {cout_A}")
print(f"Modèle B : {cout_B}")
print()

if cout_A < cout_B:
    print(f"✅ Le Modèle A est meilleur (coût plus bas de {cout_B - cout_A})")
else:
    print(f"✅ Le Modèle B est meilleur (coût plus bas de {cout_A - cout_B})")

Modèle A : 350
Modèle B : 150

✅ Le Modèle B est meilleur (coût plus bas de 200)


## 5. Synthèse

### Ce qu'on a construit
Une fonction `business_cost(y_true, y_pred)` qui calcule un coût métier custom :
- **Pénalise 10× plus** les FN (mauvais clients acceptés) que les FP (bons clients refusés)
- Reflète la réalité business : perdre du capital coûte plus cher que manquer des intérêts

### Pourquoi c'est important
Sans ce score métier, on serait tenté d'optimiser des métriques classiques (accuracy, F1) 
qui ne reflètent pas les vrais enjeux financiers. Avec ce score, on peut **comparer 
objectivement** les modèles selon leur impact réel sur la banque.

### Prochaine étape
- Sauvegarder cette fonction dans `src/models/business_score.py` pour qu'elle soit 
  réutilisable par l'API et les scripts d'entraînement.
- Utiliser cette métrique pour évaluer nos modèles (Phase 4).